In [ ]:
import os
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ROOT = os.path.dirname(os.getcwd())
TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "bronze_kafka_stream")

builder = (
    SparkSession.builder.appName("Query_Bronze")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# %%
df = spark.read.format("delta").load(TARGET_PATH)
latest_ingested_records = df.orderBy(F.col("_ingested_at").desc())

# 3. Render it to your monitor screen
latest_ingested_records.select("kafka_key", "value", "partition", "offset", "_ingested_at").show(5, truncate=False)

In [ ]:
# Load Delta Table Reference and Record Count

df = spark.read.format("delta").load(TARGET_PATH)
delta_table = DeltaTable.forPath(spark, TARGET_PATH)

print(f"Total Bronze Records Ingested: {df.count():,}")